# 01 · Baseline — Clasificación de prendas (Caso A, Visión)

**Equipo:** Francisco Arancibia · Benjamín Contreras — Machine Learning (TIEL26)

Transfer learning con **MobileNetV2** preentrenado en ImageNet sobre **Fashion-MNIST** (10 clases). Este notebook reproduce el baseline y sus métricas. Funciona en Google Colab (CPU basta) o en local.

## 1. Dependencias y datos

In [ ]:
!pip -q install torch torchvision onnx onnxruntime scikit-learn pillow psutil matplotlib

In [ ]:
import os, pathlib, subprocess
os.makedirs('data/fashion', exist_ok=True)
base = 'https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/'
for f in ['train-images-idx3-ubyte','train-labels-idx1-ubyte','t10k-images-idx3-ubyte','t10k-labels-idx1-ubyte']:
    p = f'data/fashion/{f}.gz'
    if not os.path.exists(p):
        subprocess.run(['wget','-q','-O',p, base+f+'.gz'], check=True)
sorted(os.listdir('data/fashion'))

## 2. Entrenamiento del baseline
Hiperparámetros: imágenes reescaladas a 96×96 (3 canales), 12.000 imágenes de entrenamiento, 2 épocas, AdamW + OneCycle (lr 1e-3), label smoothing 0,1, semilla 42. Sin `--weights-path` el script descarga los pesos ImageNet oficiales de torchvision.

In [ ]:
!python scripts/train_baseline.py --epochs 2 --train-samples 12000 --img-size 96

## 3. Métricas

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
m = json.load(open('reports/baseline_metrics.json'))
print(f"Exactitud: {m['accuracy']:.4f} | F1 macro: {m['f1_macro']:.4f} | tamaño: {m['tamano_mb']} MB")
cm = np.array(m['matriz_confusion'])
clases = ['camiseta','pantalon','sueter','vestido','abrigo','sandalia','camisa','zapatilla','bolso','botin']
plt.figure(figsize=(6,5)); plt.imshow(cm, cmap='Blues'); plt.colorbar()
plt.xticks(range(10), clases, rotation=60); plt.yticks(range(10), clases)
plt.xlabel('Predicción'); plt.ylabel('Real'); plt.title('Matriz de confusión (test)'); plt.tight_layout(); plt.show()

## 4. Exportación a ONNX, cuantización INT8 y benchmark (adelanto del Hito 2)

In [ ]:
!python scripts/export_onnx.py
!python scripts/benchmark.py --requests 300

In [ ]:
b = json.load(open('reports/benchmark.json'))
for r in b['resultados']:
    print(f"{r['backend']:13s} acc={r['accuracy']:.4f} p50={r['p50_ms']}ms p95={r['p95_ms']}ms rps={r['throughput_rps']} RAM={r['ram_proceso_mb']}MB tam={r['tamano_mb']}MB")